In [ ]:
# ===== CELL 1: Setup All Explainers =====
import torch
from PIL import Image
import matplotlib.pyplot as plt
import shap
print("✅ Explainability Demo")


In [ ]:
# ===== CELL 2: Load Models & Sample Data =====
# Load your 3 models from models/
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

sample_text = "This fake news article claims election fraud with manipulated video evidence"
sample_image = Image.open("data/images/test/fake/sample.jpg")
sample_video_frames = torch.randn(1, 32, 3, 299, 299).cuda()  # Sample frames


In [ ]:
# ===== CELL 3: Text SHAP =====
tokenizer = DistilBertTokenizer.from_pretrained("models/text_distilbert")
text_model = DistilBertForSequenceClassification.from_pretrained("models/text_distilbert")

def text_predict(texts):
    inputs = tokenizer(texts, return_tensors='pt', padding=True, truncation=True)
    outputs = text_model(**inputs)
    return torch.softmax(outputs.logits, dim=1)[:,1].detach().cpu().numpy()

explainer = shap.KernelExplainer(text_predict, [sample_text])
shap_values = explainer.shap_values([sample_text])

plt.figure()
shap.plots.text(shap_values[0][0], tokenizer(sample_text)['input_ids'])
plt.savefig("outputs/shap_plots/text_explain.png")
plt.show()


In [ ]:
# ===== CELL 4: Image Grad-CAM =====
from pytorch_grad_cam import GradCAM
from torchvision import transforms

image_model = EfficientNet.from_pretrained('efficientnet-b4')
image_model.load_state_dict(torch.load("models/image_efficientnet_b4.pth"))
image_model.eval()

transform = transforms.Compose([
    transforms.Resize((380, 380)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

target_layers = [image_model._conv_head]
cam = GradCAM(model=image_model, target_layers=target_layers)

input_tensor = transform(sample_image).unsqueeze(0).to(device)
grayscale_cam = cam(input_tensor=input_tensor, targets=[ClassifierOutputTarget(1)])
visualization = show_cam_on_image(np.float32(sample_image)/255., grayscale_cam[0])

plt.imshow(visualization)
plt.title("Grad-CAM: Fake Regions")
plt.axis('off')
plt.savefig("outputs/heatmaps/image_explain.png")
plt.show()


In [ ]:
# ===== CELL 5: Video Temporal Analysis =====
video_model = VideoModel()
video_model.load_state_dict(torch.load("models/video_xception_bilstm.pth"))
video_model.eval()

with torch.no_grad():
    video_pred = video_model(sample_video_frames)
    temporal_scores = []  # Per-frame scores for anomaly detection
    
    for t in range(sample_video_frames.shape[1]):
        frame_pred = video_model.spatial(sample_video_frames[:, t])  # Pseudo
        temporal_scores.append(frame_pred.max().item())
    
plt.figure(figsize=(12, 4))
plt.plot(temporal_scores)
plt.title("Temporal Anomaly Scores (Video)")
plt.xlabel("Frame Index")
plt.ylabel("Fake Score")
plt.savefig("outputs/heatmaps/video_temporal.png")
plt.show()


In [ ]:
# ===== CELL 6: Complete Multi-Modal Report =====
text_prob = 0.85  # From SHAP
image_prob = 0.92  # From GradCAM
video_prob = 0.95  # From temporal

final_prob = 0.3*text_prob + 0.3*image_prob + 0.4*video_prob
print(f"🧠 Multi-Modal Prediction: {final_prob:.3f} (FAKE)")

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].text(0.5, 0.5, f"Text\n{f"text_prob:.2f}", ha='center', va='center', fontsize=20)
axes[1].imshow(sample_image)
im = axes[2].imshow(Image.fromarray(visualization))
axes[3].plot(temporal_scores[:20])
plt.savefig("outputs/complete_explainability.png")
plt.show()
print("✅ Full explainability demo complete!")
